# 11 - A PV Fleet with Smart Inverters and Moving Cloud Shadows

Rooftop PV is not one big generator. Each system has its own orientation, inverter, and temperature,
and clouds cross a feeder as a shadow that reaches neighbouring roofs at slightly different times.
This notebook subclasses `VectorizedContainerAdapter` to model a fleet of 500 to 2,000 smart-inverter
systems with:

1. **Per-unit physics**: peak kWp, orientation and tilt factor, DC/AC ratio, NOCT cell temperature
   with a power temperature coefficient, and an inverter efficiency curve.
2. **AC clipping** at the inverter rating $S_{ac}$ (DC/AC ratio 1.1 to 1.3).
3. **Spatially correlated clouds**: a 1-D random field along the feeder, advected at speed $v$ and
   sampled at each house's position with `np.interp` at $x_i - v\,t$. Neighbours see nearly the same
   shade, distant houses do not, so the fleet ramps smoothly as N grows.
4. **Grid-code curves**: volt-watt (active-power limit) and volt-var (reactive-power response) as
   `Curve1D` of the local voltage `v_pu`, which can be a scalar or one value per house.
5. **Apparent-power limit** with either **P priority** or **Q priority**.

Outputs per fleet: `p_kw`, `q_kvar`, `curtailment_kw`, and `clipping`.

The inverter curves are illustrative (IEEE 1547-style shapes), not the parameters of a specific grid code.
Check the values against the grid code that applies to your site.

## Step 0: Imports and constants

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from energysim import world
from energysim.container import VectorizedContainerAdapter
from energysim.models.common import Curve1D

N_PV_DEMO = 1000        # houses for the single-run plots
FEEDER_M = 4000.0       # feeder length the houses are spread along (m)
HOUR = 3600.0
T0_S = 8 * 3600.0       # simulated window: 08:00 to 16:00
T1_S = 16 * 3600.0
MICRO_S = 60.0          # fleet and signal step (s)

## Step 1: The grid-code and efficiency curves

Three `Curve1D` objects are the core of this model:

- **Volt-watt**: the active-power limit as a fraction of $S_{ac}$. It is 1.0 up to 1.05 pu, then falls to
  0.2 at 1.10 pu.
- **Volt-var**: reactive power as a fraction of $S_{ac}$, positive for injection. A dead band from 0.98 to
  1.02 pu, then absorption that reaches -0.44 at 1.08 pu.
- **Inverter efficiency** as a function of the load ratio $P_{dc}/S_{ac}$. It is low at light load and
  peaks near 97 % at half load.

In [ ]:
VOLT_WATT = Curve1D(x=[0.90, 1.03, 1.05, 1.10, 1.20], y=[1.0, 1.0, 1.0, 0.2, 0.2],
                    name="volt-watt P limit (fraction of S_ac)")
VOLT_VAR = Curve1D(x=[0.92, 0.98, 1.02, 1.08], y=[0.44, 0.0, 0.0, -0.44],
                   name="volt-var Q/S_ac (+ injects, - absorbs)")
INV_EFF = Curve1D(x=[0.0, 0.05, 0.10, 0.20, 0.30, 0.50, 0.75, 1.00],
                  y=[0.80, 0.88, 0.93, 0.955, 0.965, 0.970, 0.968, 0.960],
                  name="inverter efficiency vs load ratio")

fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
v_axis = np.linspace(0.90, 1.20, 301)
axes[0].plot(v_axis, VOLT_WATT(v_axis), color="tab:red")
axes[0].set_xlabel("local voltage (pu)")
axes[0].set_ylabel("active-power limit (fraction of S_ac)")
axes[0].set_title("Volt-watt curve")
axes[0].grid(alpha=0.3)

axes[1].plot(v_axis, VOLT_VAR(v_axis), color="tab:blue")
axes[1].axhline(0, color="k", lw=0.8)
axes[1].set_xlabel("local voltage (pu)")
axes[1].set_ylabel("reactive power Q / S_ac (+ inject)")
axes[1].set_title("Volt-var curve")
axes[1].grid(alpha=0.3)

load = np.linspace(0.0, 1.2, 241)
axes[2].plot(load, INV_EFF(load), color="tab:green")
axes[2].set_xlabel("load ratio P_dc / S_ac (-)")
axes[2].set_ylabel("inverter efficiency (-)")
axes[2].set_title("Inverter efficiency curve")
axes[2].grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Step 2: The fleet class `SmartInverterPVFleet`

**Cloud field.** The feeder field is a periodic random process of length 60 km with a 10 m grid. It is white
noise smoothed by a Gaussian kernel with correlation length $\lambda$ (300 m by default, via a circular FFT),
standardised, and turned into shade by a sigmoid so that a chosen fraction of the field is covered. House $i$
at position $x_i$ sees the transmittance at $x_i - v\,(t - t_0)$, read with `np.interp`. Because the field is
periodic, the shade never runs out during a long run.

**Per step, in order.** Clear-sky horizontal irradiance is multiplied by the orientation factor and the
cloud transmittance. The cell temperature (NOCT) and DC power follow. Inverter efficiency acts on the load
ratio, and the result is capped at $S_{ac}$. Volt-watt then limits the active power. Volt-var requests
reactive power, and the apparent-power limit resolves the trade-off between P and Q.

In [ ]:
class SmartInverterPVFleet(VectorizedContainerAdapter):
    """N rooftop PV systems with per-unit physics, a moving cloud field and smart-inverter functions."""

    units = {
        "irradiance_w_m2": "W/m2", "t_ambient_degc": "degC", "v_pu": "pu",
        "p_kw": "kW", "q_kvar": "kvar", "curtailment_kw": "kW", "clipping": "pu",
        "p_dc_kw": "kW", "t_cell_degc": "degC",
    }

    def __init__(self, name="pv_fleet", size=N_PV_DEMO, step_size=MICRO_S, seed=7,
                 feeder_m=FEEDER_M, cloud_depth=0.8, cloud_fraction=0.3, cloud_speed_mps=8.0,
                 cloud_corr_m=300.0, volt_watt=True, volt_var=True, q_priority=False, t0=0.0):
        super().__init__(name=name, size=size, step_size=step_size)
        rng = np.random.default_rng(seed)
        n = self.size

        # Per-unit physics
        self.peak_kw = rng.uniform(3.0, 8.0, n)                       # DC peak (kWp)
        self.s_ac = self.peak_kw / rng.uniform(1.1, 1.3, n)           # inverter rating S_ac (kVA)
        tilt = rng.uniform(10.0, 35.0, n)
        azimuth_dev = rng.uniform(-60.0, 60.0, n)                     # degrees away from south
        self.orient = 1.0 - 0.10 * (azimuth_dev / 60.0) ** 2 - 0.04 * ((tilt - 30.0) / 20.0) ** 2
        self.noct = rng.uniform(43.0, 47.0, n)                        # nominal operating cell temp (degC)
        self.x_m = rng.uniform(0.0, feeder_m, n)                      # position along the feeder (m)
        self.gamma = -0.0035                                          # power temperature coefficient (1/K)
        self.pr = 0.92                                                # soiling, wiring, mismatch

        # Controls
        self.volt_watt = bool(volt_watt)
        self.volt_var = bool(volt_var)
        self.q_priority = bool(q_priority)

        # Cloud field: periodic, correlated, advected
        self.cloud_speed = float(cloud_speed_mps)
        self.cloud_depth = float(cloud_depth)
        self.t0 = float(t0)
        dx, length = 10.0, 60000.0
        m = int(length / dx)
        white = rng.standard_normal(m)
        k = np.arange(m)
        dist = np.minimum(k, m - k) * dx
        kernel = np.exp(-0.5 * (dist / cloud_corr_m) ** 2)
        field = np.real(np.fft.ifft(np.fft.fft(white) * np.fft.fft(kernel)))
        field = (field - field.mean()) / field.std()
        threshold = np.quantile(field, 1.0 - cloud_fraction)
        cover = 1.0 / (1.0 + np.exp(-(field - threshold) / 0.2))      # 0 clear ... 1 covered, soft edges
        trans = 1.0 - self.cloud_depth * cover
        self._cloud_len = length
        self._cloud_x = dx * np.arange(m + 1)
        self._cloud_t = np.append(trans, trans[0])                    # periodic closure for np.interp

        # States
        self.register_state("p_kw", 0.0)
        self.register_state("q_kvar", 0.0)
        self.register_state("curtailment_kw", 0.0)
        self.register_state("clipping", 0.0)
        self.register_state("p_dc_kw", 0.0)
        self.register_state("t_cell_degc", 25.0)

        # Inputs (scalar or per-unit vector)
        self.register_input("irradiance_w_m2", 0.0)
        self.register_input("t_ambient_degc", 25.0)
        self.register_input("v_pu", 1.0)

    def cloud_transmittance(self, time):
        """Transmittance at each house: the field moves at cloud_speed, so house i sees x_i - v (t - t0)."""
        x = np.mod(self.x_m - self.cloud_speed * (time - self.t0), self._cloud_len)
        return np.interp(x, self._cloud_x, self._cloud_t)

    def step(self, time, step_size=None):
        g_h = np.maximum(0.0, self.input_values["irradiance_w_m2"])
        t_amb = self.input_values["t_ambient_degc"]
        v = self.input_values["v_pu"]

        # DC side: orientation, clouds, cell temperature, power temperature coefficient
        g_poa = g_h * self.orient * self.cloud_transmittance(time)
        t_cell = t_amb + (self.noct - 20.0) / 800.0 * g_poa
        p_dc = np.maximum(0.0, self.peak_kw * (g_poa / 1000.0) * (1.0 + self.gamma * (t_cell - 25.0)) * self.pr)

        # Inverter: efficiency, then AC clipping at S_ac
        p_pot = p_dc * INV_EFF(p_dc / self.s_ac)
        clipping = p_pot > self.s_ac
        p_cap = np.minimum(p_pot, self.s_ac)

        # Volt-watt limit, then the reactive-power request from volt-var
        p_lim = np.minimum(p_cap, VOLT_WATT(v) * self.s_ac) if self.volt_watt else p_cap
        q_cmd = VOLT_VAR(v) * self.s_ac if self.volt_var else np.zeros(self.size)

        # Apparent-power limit: P priority keeps P and gives Q the remainder, Q priority does the reverse
        if self.q_priority:
            q = np.clip(q_cmd, -self.s_ac, self.s_ac)
            p = np.minimum(p_lim, np.sqrt(np.maximum(0.0, self.s_ac ** 2 - q ** 2)))
        else:
            p = p_lim
            q_max = np.sqrt(np.maximum(0.0, self.s_ac ** 2 - p ** 2))
            q = np.clip(q_cmd, -q_max, q_max)

        self.states["p_dc_kw"][:] = p_dc
        self.states["t_cell_degc"][:] = t_cell
        self.states["p_kw"][:] = p
        self.states["q_kvar"][:] = q
        self.states["curtailment_kw"][:] = np.maximum(0.0, p_pot - p)   # generation not injected
        self.states["clipping"][:] = clipping.astype(float)


OUTPUTS_PV = ["total_p_kw", "total_q_kvar", "total_curtailment_kw", "mean_clipping", "total_p_dc_kw"]


def clear_sky_w_m2(t):
    """Clear-sky horizontal irradiance, peak 950 W/m2 at solar noon."""
    h = t / HOUR
    return 950.0 * np.maximum(0.0, np.sin(np.pi * (h - 6.0) / 12.0))


def grid_voltage_pu(t):
    """Feeder voltage: a midday rise from reverse power flow, peaking at about 1.075 pu near 12:30."""
    h = t / HOUR
    return 1.0 + 0.075 * np.exp(-0.5 * ((h - 12.5) / 1.5) ** 2)


def run_pv(n=N_PV_DEMO, volt_watt=True, volt_var=True, q_priority=False, cloud_depth=0.8, seed=7):
    """Simulate n PV systems from 08:00 to 16:00; returns (results, fleet)."""
    sim = world(start_time=T0_S, stop_time=T1_S, t_macro=MICRO_S)
    sim.add_signal("irr", lambda t: clear_sky_w_m2(t), step_size=MICRO_S, units={"y": "W/m2"})
    sim.add_signal("tamb", 25.0, step_size=MICRO_S, units={"y": "degC"})
    sim.add_signal("volt", lambda t: grid_voltage_pu(t), step_size=MICRO_S, units={"y": "pu"})
    fleet = SmartInverterPVFleet(name="pv", size=n, step_size=MICRO_S, seed=seed, t0=T0_S,
                                 cloud_depth=cloud_depth, volt_watt=volt_watt, volt_var=volt_var,
                                 q_priority=q_priority)
    sim.add_fleet(sim_name="pv", container=fleet, step_size=MICRO_S, outputs=OUTPUTS_PV)
    sim.add_connections({"irr.y": "pv.irradiance_w_m2", "tamb.y": "pv.t_ambient_degc", "volt.y": "pv.v_pu"})
    return sim.simulate(pbar=False), fleet


def energy_kwh(series):
    """Integrate a kW series over its time index (hours) with the trapezoid rule."""
    trapz = getattr(np, "trapezoid", None) or np.trapz
    return float(trapz(series.to_numpy(float), series.index.to_numpy(float) / HOUR))

## Step 3: Unit test the curves with a prescribed voltage ramp

Before the grid runs, we check the curves. A small fleet with no clouds runs at 1000 W/m2, and the local
voltage ramps from 0.98 to 1.10 pu one point at a time with `set_value` followed by `step()`. We do this
twice, once with P priority and once with Q priority.

Expected behaviour: Q is zero inside the dead band, then absorbs as the voltage rises. P falls from 1.05
pu once volt-watt bites. While the inverter is clipping at S_ac, P priority keeps P at S_ac and leaves no
room for reactive power, so Q stays at zero. Q priority gives up a little P to deliver the requested vars.

In [ ]:
def voltage_ramp(q_priority):
    unit = SmartInverterPVFleet(name="ut", size=5, seed=7, cloud_depth=0.0, q_priority=q_priority)
    unit.set_value(["irradiance_w_m2"], [1000.0])
    unit.set_value(["t_ambient_degc"], [15.0])        # cooler cells, so the high DC/AC units clip at 1000 W/m2
    rows = []
    for v in np.linspace(0.98, 1.10, 25):
        unit.set_value(["v_pu"], [v])
        unit.step(12 * HOUR, MICRO_S)
        rows.append({
            "v_pu": v,
            "p_kw": float(unit.states["p_kw"].sum()),
            "q_kvar": float(unit.states["q_kvar"].sum()),
            "curtailment_kw": float(unit.states["curtailment_kw"].sum()),
            "s_kva": float(unit.s_ac.sum()),
        })
    return pd.DataFrame(rows)


ramp_p = voltage_ramp(q_priority=False)
ramp_q = voltage_ramp(q_priority=True)
display(ramp_p.round(2).iloc[::6])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))
ax1.plot(ramp_p["v_pu"], ramp_p["p_kw"], "o-", ms=3, label="P priority")
ax1.plot(ramp_q["v_pu"], ramp_q["p_kw"], "s--", ms=3, label="Q priority")
ax1.set_xlabel("local voltage (pu)")
ax1.set_ylabel("active power, 5 units (kW)")
ax1.set_title("Volt-watt in action: P during a 0.98 to 1.10 pu ramp")
ax1.grid(alpha=0.3)
ax1.legend()

ax2.plot(ramp_p["v_pu"], ramp_p["q_kvar"], "o-", ms=3, label="P priority")
ax2.plot(ramp_q["v_pu"], ramp_q["q_kvar"], "s--", ms=3, label="Q priority")
ax2.axhline(0, color="k", lw=0.8)
ax2.set_xlabel("local voltage (pu)")
ax2.set_ylabel("reactive power, 5 units (kvar)")
ax2.set_title("Volt-var in action: absorption above 1.02 pu")
ax2.grid(alpha=0.3)
ax2.legend()
plt.tight_layout()
plt.show()

## Step 4: Aggregate ramp rates versus fleet size

The cloud shadows cover the feeder, but a single house can see a sudden drop. The fleet total smooths
out as more houses are spread along the same feeder: their shade times are different, so the drops
overlap. We run three fleet sizes with the grid functions switched off, so only the clouds act, and
express the ramp rate in percent of installed AC capacity per minute.

In [ ]:
ramp_rows = []
ramp_traces = {}
for n_sz in [50, 500, 2000]:
    res_n, fleet_n = run_pv(n=n_sz, volt_watt=False, volt_var=False, cloud_depth=0.8)
    p_tot = res_n["pv.total_p_kw"]
    p_ac_tot = float(fleet_n.s_ac.sum())
    ramp_pct_min = 100.0 * np.diff(p_tot.to_numpy(float)) / (MICRO_S / 60.0) / p_ac_tot
    ramp_traces[n_sz] = (p_tot.index.to_numpy(float) / HOUR, p_tot.to_numpy(float) / p_ac_tot)
    ramp_rows.append({
        "houses N": n_sz,
        "installed AC (MVA)": round(p_ac_tot / 1000.0, 3),
        "p95 |ramp| (% of AC per min)": round(float(np.percentile(np.abs(ramp_pct_min), 95)), 2),
        "max |ramp| (% of AC per min)": round(float(np.max(np.abs(ramp_pct_min))), 2),
    })
ramp_stats = pd.DataFrame(ramp_rows)
display(ramp_stats)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))
for n_sz, (th, pn) in ramp_traces.items():
    ax1.plot(th, pn * 100.0, label=f"N = {n_sz}")
ax1.set_xlabel("time of day (h)")
ax1.set_ylabel("fleet output (% of installed AC)")
ax1.set_title("Cloud passages smooth out with fleet size")
ax1.grid(alpha=0.3)
ax1.legend()

ax2.plot(ramp_stats["houses N"], ramp_stats["p95 |ramp| (% of AC per min)"], "o-", label="p95 ramp")
ax2.plot(ramp_stats["houses N"], ramp_stats["max |ramp| (% of AC per min)"], "s--", label="max ramp")
ax2.set_xscale("log")
ax2.set_xlabel("houses N (log scale)")
ax2.set_ylabel("ramp rate (% of installed AC per minute)")
ax2.set_title("Ramp-rate statistics versus fleet size")
ax2.grid(alpha=0.3, which="both")
ax2.legend()
plt.tight_layout()
plt.show()

## Step 5: Curtailed energy: volt-watt versus no grid function

With the feeder voltage rising to about 1.075 pu around midday, the volt-watt function starts cutting active
power at 1.05 pu. We compare two runs with the same 1000 houses and the same clouds. The first has no grid
function, so the only curtailment is AC clipping at the inverter rating. The second has volt-watt and
volt-var enabled with P priority. The area under the curtailment curve is the energy the grid function
gives up.

In [ ]:
res_none, fleet_none = run_pv(n=N_PV_DEMO, volt_watt=False, volt_var=False)
res_vw, fleet_vw = run_pv(n=N_PV_DEMO, volt_watt=True, volt_var=True)

curt_none = res_none["pv.total_curtailment_kw"]
curt_vw = res_vw["pv.total_curtailment_kw"]
e_none = energy_kwh(curt_none)
e_vw = energy_kwh(curt_vw)
e_gen_none = energy_kwh(res_none["pv.total_p_kw"])
e_gen_vw = energy_kwh(res_vw["pv.total_p_kw"])
curtail_summary = pd.DataFrame({
    "no grid function": [e_none, e_gen_none],
    "volt-watt + volt-var": [e_vw, e_gen_vw],
}, index=["curtailed energy (kWh)", "exported energy (kWh)"])
display(curtail_summary.round(1))
print(f"Extra curtailment from the grid functions: {e_vw - e_none:.1f} kWh "
      f"({100.0 * (e_vw - e_none) / max(e_none, 1e-9):.0f} % of the clipping-only loss)")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))
th = curt_none.index.to_numpy(float) / HOUR
ax1.plot(th, curt_none.to_numpy(float), label="no grid function (clipping only)")
ax1.plot(th, curt_vw.to_numpy(float), label="volt-watt + volt-var")
ax1.plot(th, res_vw["pv.total_p_kw"].to_numpy(float), color="k", alpha=0.4, lw=1, label="exported (volt-watt case)")
ax1.set_xlabel("time of day (h)")
ax1.set_ylabel("fleet power (kW)")
ax1.set_title("Curtailment over the day")
ax1.grid(alpha=0.3)
ax1.legend()

ax2.plot(th, res_vw["pv.mean_clipping"].to_numpy(float) * 100.0, label="houses clipping at S_ac (%)")
ax2.plot(th, res_vw["pv.total_q_kvar"].to_numpy(float) / (fleet_vw.s_ac.sum()) * 100.0,
         label="fleet Q as % of installed S_ac")
ax2.set_xlabel("time of day (h)")
ax2.set_ylabel("percent")
ax2.set_title("Clipping and reactive support (volt-watt + volt-var run)")
ax2.grid(alpha=0.3)
ax2.legend()
plt.tight_layout()
plt.show()

## Step 6: Reactive power capability

For an apparent-power limit $S$, the reactive capability of a unit at active power $P$ is
$|Q| \le \sqrt{S^2 - P^2}$. Summed over the fleet, the envelope is $\sqrt{S_{tot}^2 - P_{tot}^2}$. The plot
shows the operating points of the volt-var run against that envelope. Points inside the envelope are
feasible. Points on the envelope have the inverter at its full apparent-power limit. Operation at high
voltage moves the operating point down to absorb reactive power, and the curtailment moves it left.

In [ ]:
s_tot = float(fleet_vw.s_ac.sum())
p_axis = np.linspace(0.0, s_tot, 400)
q_env = np.sqrt(np.maximum(0.0, s_tot ** 2 - p_axis ** 2))

fig, ax = plt.subplots(figsize=(7.5, 6))
ax.plot(p_axis, q_env, "k--", lw=1, label="capability envelope +/- sqrt(S^2 - P^2)")
ax.plot(p_axis, -q_env, "k--", lw=1)
sc = ax.scatter(res_vw["pv.total_p_kw"].to_numpy(float), res_vw["pv.total_q_kvar"].to_numpy(float),
                c=res_vw["pv.total_p_kw"].index.to_numpy(float) / HOUR, cmap="plasma", s=18)
fig.colorbar(sc, ax=ax, label="time of day (h)")
ax.axhline(0, color="grey", lw=0.6)
ax.axvline(0, color="grey", lw=0.6)
ax.set_xlabel("active power, fleet total (kW)")
ax.set_ylabel("reactive power, fleet total (kvar)")
ax.set_title(f"Reactive capability of {N_PV_DEMO:,} smart inverters (S_tot = {s_tot / 1000:.2f} MVA)")
ax.grid(alpha=0.3)
ax.legend(loc="lower left")
plt.tight_layout()
plt.show()

## Where to go next

- **Notebook 10** applies the same subclassing pattern to heat pumps and houses, with a real thermal state.
- **Notebook 06** shows the plain vectorized PV and battery containers, which this fleet extends.
- **Notebook 12** closes the loop: the same kind of fleet is placed on a real pandapower feeder, so the
  voltage that drives volt-watt and volt-var is the solved bus voltage, not a prescribed ramp.

To make this realistic, use the grid code of your site for the curves, fit the cloud correlation length and
speed to irradiance data from a local sensor network, and measure the inverter efficiency curve from the
manufacturer's test report.